In [5]:
import coremltools as ct
from transformers import AutoTokenizer, LlamaForCausalLM
import numpy as np
from src.model import KvCacheStateLlamaForCausalLM
from src.converter import LlamaCoreMLConverter
import os

RuntimeError: Failed to import transformers.models.llama.modeling_llama because of the following error (look up to see its traceback):
module 'torch' has no attribute 'version'

In [12]:
# print virtual environment
print(os.environ["VIRTUAL_ENV"])
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

/Users/danielnewman/.virtualenvs/avi_venv


In [13]:


model_id = "meta-llama/Llama-3.2-1B-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [14]:
model = LlamaForCausalLM.from_pretrained(model_id)

model = KvCacheStateLlamaForCausalLM(
    model_id,
    batch_size=1,
    context_size=2048
)

converter = LlamaCoreMLConverter(
    model,
    batch_size=1,
    context_size=2048
)

mlmodel = converter.convert(quantize=False)


/Users/danielnewman/.virtualenvs/avi_venv/lib/python3.11/site-packages/transformers/modeling_utils.py:5006: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
Torch var valueCache is added again.
Torch var keyCache is added again.
Running MIL default pipeline:  69%|██████▉   | 61/88 [00:36<01:06,  2.45s/ passes]/Users/danielnewman/code/avi/coremltools/coremltools/converters/mil/mil/passes/defs/optimize_repeat_ops.py:433: RuntimeWarning: overflow encountered in cast
  max(cur_range.low, tmp_range.low), min(cur_range.high, tmp_range.high)
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 58.45 passes/s]


RuntimeError: BlobWriter not loaded

In [ ]:
mlmodel

In [ ]:
mlmodel.save("./models/llama-3.2-1b-instruct.mlpackage")

In [ ]:

model = ct.models.MLModel('./models/llama-3.2-1b-instruct.mlpackage')

In [36]:
def _sample_token(logits, temperature):
    """
    Sample next token from logits using temperature
    """
    if temperature == 0:
        return int(np.argmax(logits))
        
    probs = np.exp(logits / temperature)
    probs = probs / np.sum(probs)
    return int(np.random.choice(len(probs), p=probs))

In [37]:
prompt = "Hello, how are you?"
inputs = tokenizer(prompt, return_tensors="np", padding=True)
input_ids = inputs["input_ids"].astype(np.int32)
current_input_ids = input_ids
generated_tokens = []

In [ ]:
inputs = tokenizer(prompt, return_tensors="np", padding=True)
input_ids = inputs["input_ids"].astype(np.int32)
print(inputs.keys())
print(input_ids)
causal_mask = inputs["attention_mask"].astype(np.int32)
print(causal_mask)

In [ ]:
# Reshape to the specified shapes in the model spec
key_cache = np.zeros((1, 32, 2048, 128), dtype=np.float16)
value_cache = np.zeros((1, 32, 2048, 128), dtype=np.float16)


# Reshape inputs as before
current_input_ids = np.array([128000, 9906, 11, 1268, 527, 499, 30]).reshape(1, -1)
causal_mask = np.ones((1, 1, len(current_input_ids[0]), len(current_input_ids[0])), dtype=np.float16)


from coremltools.models.model import MLState
state = MLState({
    'key_cache': key_cache,
    'value_cache': value_cache
})

print(type(state))

# Run inference with KV cache states

predictions = model.predict({
    'inputIds': current_input_ids.astype(np.int32),
    'causalMask': causal_mask.astype(np.float16)
}, state=state)


print(predictions)


In [ ]:
spec = model.get_spec()
spec

In [ ]:
print(model)